In [17]:
# integrate BigQuery into python
from google.cloud import bigquery 

bigquery_client = bigquery.Client(project='vf-de-aib-prd-cmr-chn-lab')

In [2]:
# UnityMedia Customer Analytical Record.
query_car_um= """
SELECT *
  FROM vf-de-datahub.vfde_dh_lake_aib_cmr_chn_data_v.uni_datamart_um_car
  LIMIT 1000
"""

query_um = bigquery_client.query(query_car_um)
car_um = query_um.to_dataframe()

# Mobile Car
query_car_mobile= """
SELECT *
  FROM vf-de-datahub.vfde_dh_lake_aib_cmr_chn_data_v.mob_datamart_car
  LIMIT 1000
"""

query_mobile = bigquery_client.query(query_car_mobile)
car_mobile = query_mobile.to_dataframe()


In [18]:
query_churn = """
WITH
  um_rescore_base AS (
  SELECT
    a.region,
    a.um_customer_id,
    a.cohort,
    b.flag_churn,
    case 
    when 
    case when a.internet_cons_days_out_of_contract > 0 
    then a.internet_cons_days_out_of_contract 
    else a.internet_cons_days_until_contract_end 
    end
    between 31 and 180 then 1
    else 0 
    end as Contract_ooc_ic_31_180,
    a.customer_age_class,
    a.count_internet_previous_cancellations_0_18,
    a.basket_months_since_contract_start_date,
    a.count_transactions_Online,
    a.internet_product_name,
    a.internet_diff_churn_months,
    a.internet_all_outofcontract,
    a.calls_inbound_general_concern_waiting_time_inqueue_0_6,
    a.sales_channel_first_activation,
    a.internet_months_since_contract_start_date,
    a.sales_channel_last_transaction_int_tv,
    a.internet_months_until_contract_end,
    a.internet_days_until_initial_contract_end,
    a.contract_summaries_csconfirmed_0_6,
    a.campaign_dima_mailspec_0_18,
    a.contract_summaries_csconfirmed_0_3,
    a.count_calls_inbound_general_concern_0_6,
    a.basket_package_price_list,
    a.basket_months_until_contract_end,
    a.basket_days_until_contract_end,
    a.calls_inbound_talk_time_0_6,
    a.non_internet_months_until_contract_end,
    a.internet_package_price_list,
    a.non_internet_months_since_contract_start_date,
    a.internet_package_price_disc,
    a.count_previous_cancellations_0_18,
    a.non_internet_days_until_contract_end,
    a.count_calls_inbound_0_6,
    a.calls_inbound_carecall_contract_waiting_time_inqueue_0_6,
    a.campaign_dima_service_0_6,
    a.basket_package_price_disc,
    a.basket_proportion_disc,
    a.campaign_dima_0_6,
    a.internet_product_category,
    a.non_internet_product_category,
    a.sales_channel_lvl3_last_transaction_int_tv,
    a.count_transactions_Commercial_Operations,
    a.basket_days_until_initial_contract_end,
    a.campaign_dima_service_0_3,
    a.basket_product_category,
    a.calls_inbound_waiting_time_inqueue_0_6,
    a.basket_product_prod,
    a.internet_cpe,
    a.number_cust_at_cmts_segment,
    a.internet_down_speed,
    a.campaign_dima_service_0_18,
    a.network_node_performance_days_yellow_0_1,
    a.basket_days_until_next_promo_end,
    a.campaign_dima_xsell_0_18,
    a.sales_channel_last_productchange,
    a.campaign_dima_0_18,
    a.count_transactions_other,
    a.internet_days_until_contract_end,
    a.internet_cpe_days_since_installation_date,
    a.sales_channel_lvl3_first_activation,
    a.number_cust_at_lastamp_segment,
    a.internet_cons_package_price_list,
    a.non_internet_product_prod,
    a.sales_channel_last_transaction,
    a.campaign_dima_email_0_18,
    a.number_cust_at_node_segment
  FROM
    `vf-de-datahub.vfde_dh_lake_aib_cmr_chn_data_v.uni_datamart_um_car` a
  LEFT JOIN
    `vf-de-datahub.vfde_dh_lake_aib_cmr_chn_data_v.uni_datamart_label_churn` b
  ON
    a.um_customer_id = b.um_customer_id
    AND a.cohort = b.cohort
    AND a.region = b.region
  WHERE
    a.cohort = '2024-08-01'
    AND a.internet_cons_package_count > 0
    AND a.internet_cons_days_until_contract_end <= 400 ),
  destination_table AS (
  SELECT
    *
  FROM (
    SELECT
      DISTINCT a.*
    FROM
      um_rescore_base a
    LEFT JOIN (
      SELECT
        DISTINCT a.um_customer_id,
        a.region
      FROM
        um_rescore_base a
      LEFT JOIN
        `vf-de-datahub.vfde_dh_lake_aib_cmr_chn_data_v.uni_datamart_churn_order_entry`b
      ON
        a.um_customer_id = b.um_customer_id
        AND a.region = b.region
      WHERE
        cohort = '2024-08-01'
        AND flag_churn = 0
        AND LAST_DAY(woe_date) BETWEEN LAST_DAY(DATE_ADD('2024-08-01', INTERVAL -3 MONTH), MONTH)
        AND LAST_DAY(DATE_ADD('2024-08-01', INTERVAL -1 MONTH), MONTH) ) b
    ON
      a.um_customer_id = b.um_customer_id
      AND a.region = b.region
    WHERE
      b.um_customer_id IS NULL
      AND cohort = '2024-08-01'
      AND flag_churn = 0
    UNION ALL
    SELECT
      b.*
    FROM
      um_rescore_base b
    WHERE
      cohort = '2024-08-01'
      AND flag_churn = 1 ) cv )
SELECT
  *
FROM (
  SELECT
    x.*
  FROM
    destination_table x )
"""

In [19]:
query_churn = bigquery_client.query(query_churn)
#raw data from BQ
raw_data = query_churn.to_dataframe()

In [20]:
import pandas as pd
pd.set_option('display.max_columns', None)
raw_data.head()

,region,um_customer_id,cohort,flag_churn,Contract_ooc_ic_31_180,customer_age_class,count_internet_previous_cancellations_0_18,basket_months_since_contract_start_date,count_transactions_Online,internet_product_name,internet_diff_churn_months,internet_all_outofcontract,calls_inbound_general_concern_waiting_time_inqueue_0_6,sales_channel_first_activation,internet_months_since_contract_start_date,sales_channel_last_transaction_int_tv,internet_months_until_contract_end,internet_days_until_initial_contract_end,contract_summaries_csconfirmed_0_6,campaign_dima_mailspec_0_18,contract_summaries_csconfirmed_0_3,count_calls_inbound_general_concern_0_6,basket_package_price_list,basket_months_until_contract_end,basket_days_until_contract_end,calls_inbound_talk_time_0_6,non_internet_months_until_contract_end,internet_package_price_list,non_internet_months_since_contract_start_date,internet_package_price_disc,count_previous_cancellations_0_18,non_internet_days_until_contract_end,count_calls_inbound_0_6,calls_inbound_carecall_contract_waiting_time_inqueue_0_6,campaign_dima_service_0_6,basket_package_price_disc,basket_proportion_disc,campaign_dima_0_6,internet_product_category,non_internet_product_category,sales_channel_lvl3_last_transaction_int_tv,count_transactions_Commercial_Operations,basket_days_until_initial_contract_end,campaign_dima_service_0_3,basket_product_category,calls_inbound_waiting_time_inqueue_0_6,basket_product_prod,internet_cpe,number_cust_at_cmts_segment,internet_down_speed,campaign_dima_service_0_18,network_node_performance_days_yellow_0_1,basket_days_until_next_promo_end,campaign_dima_xsell_0_18,sales_channel_last_productchange,campaign_dima_0_18,count_transactions_other,internet_days_until_contract_end,internet_cpe_days_since_installation_date,sales_channel_lvl3_first_activation,number_cust_at_lastamp_segment,internet_cons_package_price_list,non_internet_product_prod,sales_channel_last_transaction,campaign_dima_email_0_18,number_cust_at_node_segment
0,KBW,872399569,2024-08-01,0,1,25 - 29,0,19,1,GIGAZUHAUSE 250 KABEL,0,0,0,Direct,19,Direct,5,171,0,0,0,0,44.990000000,5,171,0,0,44.990000000,19,44.990000000,0,30,0,0,0,44.990000000,0E-9,0,2PLAY,SUPERWLAN,Online,0,171,0,SUPERWLAN,0,SUPERWLAN,VF-STATION WIFI6,17331,250,1,27,0,0,None,3,0,171,562,Online,27,44.990000000,SUPERWLAN,Direct,8,239
1,NRW,717758228,2024-08-01,0,1,75 - 79,0,21,0,GIGAZUHAUSE 50 TREUE C,0,0,0,None,21,UM - Sales in Service,3,114,0,0,0,0,24.990000000,3,114,0,0,24.990000000,21,24.990000000,0,30,1,0,0,24.990000000,0E-9,0,2PLAY,SUPERWLAN,Commercial Operations,2,114,0,SUPERWLAN,3058,SUPERWLAN,VODAFONE STATION,13058,50,1,27,0,0,Outbound Upsell Fixed,1,0,114,1074,None,38,24.990000000,SUPERWLAN,UM - Sales in Service,6,747
2,KBW,509317029,2024-08-01,0,1,25 - 29,0,23,0,RED INTERNET UND PHONE 250 CABLE U,0,0,0,None,23,Sonderkarten,1,38,0,3,0,0,44.980000000,1,38,0,0,44.980000000,23,44.980000000,0,30,0,0,0,44.980000000,0E-9,0,2PLAY,SUPERWLAN,Sonstige,1,38,0,SUPERWLAN,0,SUPERWLAN,VF-STATION WIFI6,21555,250,2,27,0,5,1st Level VUM Tech,2,2,38,464,None,57,44.980000000,SUPERWLAN,Sonderkarten,9,423
3,HSN,444265909,2024-08-01,0,0,50 - 54,0,150,0,2PLAY 32MBIT,0,1,0,None,150,UM - Sales in Service,0,0,0,1,0,0,67.870000000,0,30,0,0,41.890000000,150,41.890000000,0,30,0,0,0,67.870000000,0E-9,0,2PLAY,TV START,Commercial Operations,1,0,0,TV START,0,TV START,VODAFONE STATION,6304,32,1,14,0,0,None,1,0,30,1393,None,31,41.890000000,TV START,UM - Sales in Service,5,576
4,NRW,201845973,2024-08-01,0,0,45 - 49,0,16,2,VF CABLEMAX 1000,0,0,0,None,16,UM - Sales in Service,8,264,0,0,0,0,39.980000000,8,264,0,0,39.980000000,16,39.980000000,2,30,0,0,0,39.980000000,0E-9,0,2PLAY,SUPERWLAN,Commercial Operations,3,264,0,SUPERWLAN,0,SUPERWLAN,VODAFONE STATION,2519,1000,0,27,0,1,Inbound StayVF UM,0,0,264,1095,None,31,39.980000000,SUPERWLAN,UM - Sales in Service,6,116


In [21]:
data_description = {'region':'Region code depending on the database in CableMaster, where the data is extracted from',
                   'um_customer_id':'Unitymedia Customer id, only unique in combination with region.(column is anonymized)',
                   'cohort':'Monthly cohort',
                    'Contract_ooc_ic_31_180':'This variable was created. Checks whether the internet contract is due then takes the amount of days it was oversue, otherwise the day until the end of the contract and makes sure they are between 31 and 180 days',
                   'customer_age_class':'Age class of the customer.',
                    'count_internet_previous_cancellations_0_18':'Count of previous consumer internet cancellations by customer.',
                    'basket_months_since_contract_start_date':'Months since start of current contract',
                    'count_transactions_Online':'Count of transactions of Sales Channel: Online',
                    'internet_product_name':'Name of the booked product.',
                    'internet_diff_churn_months':'Months between internet contract end date and the last internet cancellation date)',
                    'internet_all_outofcontract':'Flag if all products in the relevant group are out the initial contract duration',
                    'calls_inbound_general_concern_waiting_time_inqueue_0_6':'Waiting time in inbound call queue per call reason',
                    'sales_channel_first_activation':'Sales Channel of the first activation, detail description',
                    'internet_months_since_contract_start_date':'Months since start of current contract.',
                    'sales_channel_last_transaction_int_tv':'Sales Channel detailed description of the last product change for Internet / TV, detail',
                    'internet_months_until_contract_end':'Months until end of current contract.',
                    'internet_days_until_initial_contract_end':'Days until initial contract end date.',
                    'contract_summaries_csconfirmed_0_6':'Counts accepted contract summaries.',
                    'campaign_dima_mailspec_0_18':'Count of all specifically directed mail to the customer.',
                    'contract_summaries_csconfirmed_0_3':'Counts accepted contract summaries.',
                    'count_calls_inbound_general_concern_0_6':'Count of inbound calls of customer per general concern',
                    'basket_package_price_list':'List price of the booked product.',
                    'basket_months_until_contract_end':'Months until end of current contract.',
                    'basket_days_until_contract_end':'Days until end of current contract.',
                    'calls_inbound_talk_time_0_6':'Talk time in inbound call queue per call reason',
                    'non_internet_months_until_contract_end':'Months until end of current contract.',
                    'internet_package_price_list':'List price of the booked product.',
                    'non_internet_months_since_contract_start_date':'Months since start of current contract.',
                    'internet_package_price_disc':'	Discounted price of the booked product.',
                    'count_previous_cancellations_0_18':'Count of previous cancellations by customer.',
                    'non_internet_days_until_contract_end':'Days until end of current contract.',
                    'count_calls_inbound_0_6':'Count of inbound calls of customer',
                    'calls_inbound_carecall_contract_waiting_time_inqueue_0_6':'Waiting time in inbound call queue per carecall contract reason',
                    'campaign_dima_service_0_6':'Count of all service direct marketing campaigns the customer was included.',
                    'basket_package_price_disc':'Discounted price of the booked product.',
                    'basket_proportion_disc':'Ratio discount price to list price ( 1- (package_price_disc/package_price_list) )',
                    'campaign_dima_0_6':'Count of all direct marketing campaigns the customer was included.',
                    'internet_product_category':'Category of the booked product.',
                    'non_internet_product_category':'Category of the booked product.',
                    'sales_channel_lvl3_last_transaction_int_tv':'Sales Channel descriptionof the last product change for Internet / TV',
                    'count_transactions_Commercial_Operations':'Count of transactions of Sales Channel: Commercial Operations',
                    'basket_days_until_initial_contract_end':'Days until initial contract end date.',
                    'campaign_dima_service_0_3':'Count of all service direct marketing campaigns the customer was included.',
                    'basket_product_category':'Category of the booked product.',
                    'calls_inbound_waiting_time_inqueue_0_6':'Waiting time in inbound call queue per call reason',
                    'basket_product_prod':'Alternative name of the booked product.',
                    'internet_cpe':'Internet Customer Premises Equipment (CPE).',
                    'number_cust_at_cmts_segment':'Number of customers connected to CMTS',
                    'internet_down_speed':'Booked Downspeed bandwith of internet product',
                    'campaign_dima_service_0_18':'Count of all service direct marketing campaigns the customer was included.',
                    'network_node_performance_days_yellow_0_1':'Number of days in given time frame with node performance to be considered yellow',
                    'basket_days_until_next_promo_end':'Days until the end of the next promo.',
                    'campaign_dima_xsell_0_18':'Count of all X-Sell direct marketing campaigns the customer was included.',
                    'sales_channel_last_productchange':'Sales Channel detail description of the last product change',
                    'campaign_dima_0_18':'Count of all direct marketing campaigns the customer was included.',
                    'count_transactions_other':'Count of transactions of Sales Channel: Other',
                    'internet_days_until_contract_end':'Days until end of current contract.',
                    'internet_cpe_days_since_installation_date':'Days since installation Date of internet CPE.',
                    'sales_channel_lvl3_first_activation':'	Sales Channel of the first activation, description',
                    'number_cust_at_lastamp_segment':'	Number of customers connected to the last amplifier',
                    'internet_cons_package_price_list':'List price of the booked product.',
                    'non_internet_product_prod':'Alternative name of the booked product.',
                    'sales_channel_last_transaction':'Sales Channel detail description of the last transaction',
                    'campaign_dima_email_0_18':'Count of all specifically directed E-Mail to the customer.',
                    'number_cust_at_node_segment':'Number of customers connected to Node'
                   
                    
                   }

In [22]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.DataFrame(list(data_description.items()), columns=['Column Name', 'Description'])

,Column Name,Description
0,region,"Region code depending on the database in CableMaster, where the data is extracted from"
1,um_customer_id,"Unitymedia Customer id, only unique in combination with region.(column is anonymized)"
2,cohort,Monthly cohort
3,Contract_ooc_ic_31_180,"This variable was created. Checks whether the internet contract is due then takes the amount of days it was oversue, otherwise the day until the end of the contract and makes sure they are between 31 and 180 days"
4,customer_age_class,Age class of the customer.
5,count_internet_previous_cancellations_0_18,Count of previous consumer internet cancellations by customer.
6,basket_months_since_contract_start_date,Months since start of current contract
7,count_transactions_Online,Count of transactions of Sales Channel: Online
8,internet_product_name,Name of the booked product.
9,internet_diff_churn_months,Months between internet contract end date and the last internet cancellation date)


In [23]:
#Manually selected columns (some columns werethe same but for different time period)
columns_needed = ['region', 'cohort', 'flag_churn', 'customer_age_class', 'count_internet_previous_cancellations_0_18', 'basket_months_since_contract_start_date', 
                  'count_transactions_Online', 'internet_product_name', 'internet_diff_churn_months', 'internet_all_outofcontract', 
                  'calls_inbound_waiting_time_inqueue_0_6', 'calls_inbound_talk_time_0_6', 'count_calls_inbound_0_6', 'internet_months_since_contract_start_date', 
                  'internet_months_until_contract_end', 'internet_days_until_initial_contract_end', 'campaign_dima_email_0_18', 'campaign_dima_service_0_18', 
                  'campaign_dima_xsell_0_18', 'campaign_dima_0_18', 'contract_summaries_csconfirmed_0_3', 'basket_package_price_list', 'internet_package_price_list', 
                  'internet_package_price_disc', 'basket_package_price_disc', 'internet_cons_package_price_list', 'basket_months_until_contract_end', 
                  'basket_days_until_contract_end', 'non_internet_months_until_contract_end', 'non_internet_days_until_contract_end', 'internet_days_until_contract_end', 
                  'non_internet_months_since_contract_start_date', 'count_previous_cancellations_0_18', 'basket_proportion_disc', 'internet_product_category', 
                  'non_internet_product_category', 'sales_channel_lvl3_last_transaction_int_tv', 'count_transactions_Commercial_Operations', 'basket_product_category', 
                  'internet_cpe', 'internet_down_speed', 'basket_days_until_next_promo_end', 'sales_channel_last_productchange', 'count_transactions_other', 
                  'number_cust_at_lastamp_segment', 'sales_channel_last_transaction', 'number_cust_at_node_segment']

In [24]:
# Make the data smaller for each class of churn the same amount
churn_1_sample = raw_data[raw_data['flag_churn'] == 1].sample(n=50000, random_state=42)
churn_0_sample = raw_data[raw_data['flag_churn'] == 0].sample(n=50000, random_state=42)
balanced_sample = pd.concat([churn_1_sample, churn_0_sample])
balanced_sample = balanced_sample.sample(frac=1, random_state=42).reset_index(drop=True)

In [25]:
balanced_sample = balanced_sample[columns_needed]

In [26]:
# Null values in data
null_counts = balanced_sample.isnull().sum()
null_counts

region                                               0
cohort                                               0
flag_churn                                           0
customer_age_class                                   0
count_internet_previous_cancellations_0_18           0
basket_months_since_contract_start_date              0
count_transactions_Online                            0
internet_product_name                                0
internet_diff_churn_months                           0
internet_all_outofcontract                           0
calls_inbound_waiting_time_inqueue_0_6               0
calls_inbound_talk_time_0_6                          0
count_calls_inbound_0_6                              0
internet_months_since_contract_start_date            0
internet_months_until_contract_end                   0
internet_days_until_initial_contract_end             0
campaign_dima_email_0_18                             0
campaign_dima_service_0_18                           0
campaign_d

In [27]:
balanced_data_42 = balanced_sample.dropna(axis=1)

In [34]:
balanced_data_42.to_csv('balanced_data_42.csv', index=False)

## Column Selection with Tree Importance

In [29]:
result_df_copy = balanced_data_42.copy()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import shap
import numpy as np

for col in balanced_data_manual_columns.select_dtypes(include='object').columns:
    if balanced_data_manual_columns[col].nunique()==2:
        le = LabelEncoder()
        balanced_data_manual_columns[col] = le.fit_transform(balanced_data_manual_columns[col])
    else:
        balanced_data_manual_columns = pd.get_dummies(balanced_data_manual_columns, columns=[col], drop_first=True)

X = balanced_data_manual_columns.drop(columns=['flag_churn', 'cohort'])
y = balanced_data_manual_columns['flag_churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Featuer Importance from XGBoost

In [27]:
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

important_columns_xgb = pd.DataFrame({'feature': X.columns,
                             'importance': model.feature_importances_}).sort_values(by='importance', ascending=False)['feature'].tolist()

/opt/conda/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [09:15:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [30]:
important_columns_xgb[:10]

['basket_package_price_disc_0E-9',
 'customer_age_class_25 - 29',
 'internet_package_price_disc_0E-9',
 'basket_months_until_contract_end',
 'basket_days_until_contract_end',
 'count_previous_cancellations_0_18',
 'internet_product_name_GIGAZUHAUSE 1000 KABEL',
 'internet_product_name_RED INTERNET UND PHONE 1000 CABLE U',
 'internet_diff_churn_months',
 'customer_age_class_35 - 39']

### Permutation Importance

In [36]:
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

important_columns_perm = pd.DataFrame({'feature': X.columns,
                             'importance': perm_importance.importances_mean}).sort_values(by='importance', ascending=False)['feature'].tolist()

KeyboardInterrupt: 

In [ ]:
important_columns_perm[:10]

### SHAP

In [34]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap_importance_df = pd.DataFrame({'feature': X.columns,
                             'importance': np.abs(shap_values).mean(axis=0)}).sort_values(by='importance', ascending=False)['feature'].tolist()

In [35]:
shap_importance_df[:10]

['basket_months_since_contract_start_date',
 'basket_days_until_contract_end',
 'basket_months_until_contract_end',
 'customer_age_class_25 - 29',
 'region_KBW',
 'calls_inbound_waiting_time_inqueue_0_6',
 'internet_days_until_initial_contract_end',
 'internet_months_since_contract_start_date',
 'internet_diff_churn_months',
 'internet_product_name_VF CABLEMAX 1000']

### Columns from Model Importance

In [30]:
filtered_columns = ['customer_age_class',
                   'region',
                   'count_previous_cancellations_0_18',
                   'basket_months_since_contract_start_date',
                   'basket_days_until_contract_end',
                   'basket_months_until_contract_end',
                   'basket_package_price_disc',
                   'basket_product_category',
                   'basket_package_price_list',
                   'internet_package_price_disc',
                   'internet_product_name',
                   'internet_diff_churn_months',
                   'internet_days_until_initial_contract_end',
                   'internet_months_since_contract_start_date',
                    'flag_churn']
balanced_data_15 = balanced_data_42[filtered_columns]

In [31]:
balanced_data_15.head()

,customer_age_class,region,count_previous_cancellations_0_18,basket_months_since_contract_start_date,basket_days_until_contract_end,basket_months_until_contract_end,basket_package_price_disc,basket_product_category,basket_package_price_list,internet_package_price_disc,internet_product_name,internet_diff_churn_months,internet_days_until_initial_contract_end,internet_months_since_contract_start_date,flag_churn
0,35 - 39,NRW,0,46,30,0,32.990000000,2PLAY,32.990000000,32.990000000,VF RIP 250 U-TREUE,0,0,46,0
1,18 - 24,KBW,0,20,148,4,44.990000000,SUPERWLAN,44.990000000,44.990000000,GIGAZUHAUSE 250 KABEL,0,148,20,0
2,35 - 39,KBW,4,13,334,11,104.940000000,VF PREMIUM,109.940000000,64.980000000,GIGAZUHAUSE 1000 KABEL,17,334,13,1
3,50 - 54,HSN,0,118,30,0,87.930000000,VF PREMIUM,92.930000000,46.970000000,VF RIP 250 U-TREUE,0,0,39,0
4,45 - 49,NRW,0,102,30,0,42.970000000,SUPERWLAN,42.970000000,42.970000000,RED INTERNET UND PHONE 250 CABLE U,0,0,102,0


In [39]:
balanced_data_15.to_csv('balanced_data_15.csv', index=False)

# Columns Investigation

### customer_age_class

In [32]:
#One way to handle is using them as categories
#Make them an integer and use as an integer, and when the result comes we will classifyto thiese intervals
balanced_data_15['customer_age_class'].unique()

array(['35 - 39', '18 - 24', '50 - 54', '45 - 49', '40 - 44', '70 - 74',
       '55 - 59', '60 - 64', '25 - 29', '80 and over', '30 - 34',
       '65 - 69', 'not available', '75 - 79', 'under 18'], dtype=object)

In [78]:
def get_interval_mean(age_str):
    if 'under' in age_str:
        return 17
    elif 'and over' in age_str:
        return 80
    elif 'not' in age_str:
        return None
    else:
        low, high = map(int, age_str.split(' - '))
        return (low + high) // 2

def age_class_to_int(data):
    # converts customer_age_class to customer_age
    data['customer_age'] = data['customer_age_class'].apply(get_interval_mean)
    data = data.drop(columns = ['customer_age_class'])
    return data

In [79]:
balanced_data_15_age_int = balanced_data_15.copy(deep=True)
age_class_to_int(balanced_data_15_age_int).head()


,region,count_previous_cancellations_0_18,basket_months_since_contract_start_date,basket_days_until_contract_end,basket_months_until_contract_end,basket_package_price_disc,basket_product_category,basket_package_price_list,internet_package_price_disc,internet_product_name,internet_diff_churn_months,internet_days_until_initial_contract_end,internet_months_since_contract_start_date,flag_churn,customer_age
0,NRW,0,46,699,23,38.440000000,DIGITAL KABELANSCHLUSS,38.440000000,34.990000000,RED INTERNET UND PHONE 50 CABLE U,0,0,46,0,32.0
1,NRW,0,200,30,0,69.960000000,N/A,69.960000000,44.970000000,VF RIP 100 U-TREUE,0,0,28,0,80.0
2,KBW,0,87,30,0,75.950000000,HD OPTION,75.950000000,44.980000000,VF CABLEMAX 500,0,0,53,1,47.0
3,NRW,0,114,30,0,43.950000000,3PLAY,43.950000000,43.950000000,3PLAY SMART 50,0,0,114,0,57.0
4,KBW,0,20,137,4,19.990000000,2PLAY,24.990000000,19.990000000,GIGAZUHAUSE 50 TREUE C,24,137,20,0,52.0


### internet_product_category

In [33]:
# basket_product_category is named after the most expensive service in the basekt (although there are some excheptions for example for SUPERWLAN)
balanced_data_15_internet = balanced_data_15.copy(deep=True)

In [34]:
unique_categories = balanced_data_15[['basket_product_category', 'internet_product_name']].drop_duplicates()

In [35]:
unique_categories[unique_categories['basket_product_category']=='SUPERWLAN'].head()

,basket_product_category,internet_product_name
1,SUPERWLAN,GIGAZUHAUSE 250 KABEL
4,SUPERWLAN,RED INTERNET UND PHONE 250 CABLE U
6,SUPERWLAN,RED INTERNET UND PHONE 100 CABLE U
9,SUPERWLAN,VF CABLEMAX 500
13,SUPERWLAN,GIGAZUHAUSE 100 TREUE C


In [36]:
internet_product = sorted(balanced_data_15['internet_product_name'].unique())
print(len(internet_product))
internet_product

145


['128KBIT',
 '1MBIT',
 '1PLAY 128MBIT',
 '1PLAY 30MBIT 4ALL',
 '1PLAY 32MBIT',
 '1PLAY 64MBIT',
 '1PLAY MAX 400',
 '2PLAY 100MBIT',
 '2PLAY 30MBIT 4ALL',
 '2PLAY 30MBIT 4ALL TREU',
 '2PLAY 32MBIT',
 '2PLAY 50MBIT',
 '2PLAY 64MBIT',
 '2PLAY 6MBIT',
 '2PLAY COMFORT 120',
 '2PLAY FLY 1000',
 '2PLAY FLY 400',
 '2PLAY JUMP 120',
 '2PLAY JUMP 150',
 '2PLAY JUMP 200',
 '2PLAY MAX 400',
 '2PLAY PLUS 100',
 '2PLAY PLUS 120',
 '2PLAY PLUS 50',
 '2PLAY PLUS 50 (CHECK24)',
 '2PLAY PREMIUM 100',
 '2PLAY PREMIUM 150',
 '2PLAY PREMIUM 200',
 '2PLAY PREMIUM 250',
 '2PLAY START 30',
 '2PLAY TREUE 100',
 '2PLAY TREUE 200',
 '2PLAY TREUE 250',
 '2PLAY TREUE 400',
 '2PLAY TREUE 50',
 '2PLAY TREUE 64',
 '2PLAY VORTEIL 150',
 '3PLAY 10MBIT',
 '3PLAY 25MBIT + PAY',
 '3PLAY 30MBIT 4ALL',
 '3PLAY 32MBIT',
 '3PLAY 32MBIT + HD BOX + PAY',
 '3PLAY 32MBIT + HIGHLIGHTS',
 '3PLAY 32MBIT + PAY',
 '3PLAY 32MBIT HD BOX',
 '3PLAY 32MBIT HD BOX + ALLSTARS',
 '3PLAY 50MBIT + HD BOX + PAY',
 '3PLAY 50MBIT + PAY',
 '3PLAY 5

In [37]:
import re

def remove_numbers(s):
    words = s.split()
    cleaned_words = [words[0]] + [word for word in words[1:] if not re.search(r'\d', word)]
    return ' '.join(cleaned_words)

balanced_data_15_internet['internet_product_name'] = balanced_data_15_internet['internet_product_name'].apply(lambda x: x.replace ('UND', '&') if 'RED INTERNET UND PHONE' in x else x)
balanced_data_15_internet['internet_product_name'] = balanced_data_15_internet['internet_product_name'].apply(remove_numbers)

In [38]:
internet_product = sorted(balanced_data_15_internet['internet_product_name'].unique())
print(len(internet_product))
internet_product

66


['128KBIT',
 '1MBIT',
 '1PLAY',
 '1PLAY MAX',
 '2PLAY',
 '2PLAY COMFORT',
 '2PLAY FLY',
 '2PLAY JUMP',
 '2PLAY MAX',
 '2PLAY PLUS',
 '2PLAY PREMIUM',
 '2PLAY START',
 '2PLAY TREU',
 '2PLAY TREUE',
 '2PLAY VORTEIL',
 '3PLAY',
 '3PLAY + HD BOX + PAY',
 '3PLAY + HIGHLIGHTS',
 '3PLAY + PAY',
 '3PLAY COMFORT',
 '3PLAY FLY',
 '3PLAY HD BOX',
 '3PLAY HD BOX + ALLSTARS',
 '3PLAY JUMP',
 '3PLAY MAX',
 '3PLAY PLUS',
 '3PLAY PLUS HRZ',
 '3PLAY PLUS MTA',
 '3PLAY PREMIUM',
 '3PLAY PREMIUM HRZ',
 '3PLAY PREMIUM MTA',
 '3PLAY SKY',
 '3PLAY SMART',
 '3PLAY SMART HRZ',
 '3PLAY START',
 '3PLAY TREUE',
 '3PLAY TREUE GENRE',
 '3PLAY TREUE HRZ',
 '3PLAY TREUE MODEM',
 '3PLAY VORTEIL',
 'BASIS INTERNET WOWI',
 'EAZY',
 'EAZY FLEX',
 'GIGAZUHAUSE CABLEMAX',
 'GIGAZUHAUSE CABLEMAX EB',
 'GIGAZUHAUSE CABLEMAX FF',
 'GIGAZUHAUSE KABEL',
 'GIGAZUHAUSE TREUE',
 'GIGAZUHAUSE TREUE B',
 'GIGAZUHAUSE TREUE C',
 'GIGAZUHAUSE TREUE D',
 'HSI HZ BOX ONLY',
 'INTERNET',
 'INTERNET COMFORT',
 'INTERNET PREMIUM',
 'INTER

### basket_product_category

In [39]:
# basket_product_category is named after the most expensive service in the basekt (although there are some excheptions for example for SUPERWLAN)
balanced_data_15_basket = balanced_data_15.copy(deep=True)

In [40]:
basket_product = sorted(balanced_data_15_basket['basket_product_category'].unique())
print(len(basket_product))
basket_product

60


['1PLAY',
 '2PLAY',
 '3PLAY',
 '3_HD_SKY FILM SPORT BULI HL_AS',
 'ALLSTARS',
 'ANALOG KABELANSCHLUSS',
 'ARENA',
 'AVM POWERLINE 1260E',
 'AVM REPEATER 1750E',
 'BASISPREISANPASSUNG',
 'BOOSTER RETENTION',
 'BOOSTER STANDARD',
 'CUSTOMER PROFILE',
 'DIGITAL INTERNATIONAL',
 'DIGITAL KABELANSCHLUSS',
 'DISCOUNT',
 'DOCU',
 'DYNAMIC IP DUAL STACK',
 'ENABLER KOMBI BUNDLE RABATT',
 'EUROPA FLAT PLUS',
 'GIGA TV',
 'GIGAKOMBI MARKER SERVICE',
 'GIGATV CABLE',
 'GIGATV CABLE BOX',
 'GUTSCHRIFT',
 'HD MODUL',
 'HD MODUL (MIETE)',
 'HD OPTION',
 'HD OPTION ALLSTARS',
 'HD RECEIVER',
 'HD RECORDER',
 'HIGHLIGHTS',
 'HRZ RECEIVER',
 'HRZ RECORDER',
 'HZ GO S/A',
 'HZ TV',
 'INTERNATIONAL FLAT PLUS',
 'KIDS',
 'KMV PREMIUM',
 'MIETE DIGITAL-RECEIVER',
 'MMA ACTIVE',
 'MULTIROOM OPTION',
 'N/A',
 'OTC INTERNET',
 'POSITIVOPTION 3P SMART 50',
 'SD RECEIVER',
 'SD RECORDER',
 'SECURITY  PACK',
 'SERIES AND MOVIES',
 'SKY WELT',
 'SKY WELT 3P',
 'SPAR MOBIL',
 'SPORTS',
 'SUPERWLAN',
 'SUPERWLAN MO

In [101]:
balanced_data_15_internet.to_csv('balanced_data_15_internet.csv', index=False)

# Less Data

In [42]:
# Make the data smaller for each class of churn the same amount
churn_1_sample = balanced_data_15_internet[balanced_data_15_internet['flag_churn'] == 1].sample(n=5000, random_state=42)
churn_0_sample = balanced_data_15_internet[balanced_data_15_internet['flag_churn'] == 0].sample(n=5000, random_state=42)
result_df = pd.concat([churn_1_sample, churn_0_sample])
result_df = result_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [1]:
result_df.to_csv('result_df.csv', index=False)

NameError: name 'result_df' is not defined

# Filtered from result_df

In [7]:
import pandas as pd
result_df = pd.read_csv('result_df.csv')
columns = ['customer_age_class', 'region', 'basket_package_price_disc', 'internet_package_price_disc',
          'basket_days_until_contract_end', 'count_previous_cancellations_0_18', 'internet_product_name',
          'flag_churn']
result_df_filtered = result_df.loc[:, columns]
result_df_filtered.to_csv('data.csv', index=False)



In [9]:
data = pd.read_csv('data.csv')
data.head()

,customer_age_class,region,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,70 - 74,NRW,64.95,54.97,366,0,3PLAY PREMIUM HRZ,0
1,35 - 39,HSN,44.98,44.98,100,0,RED INTERNET & PHONE CABLE U,1
2,35 - 39,NRW,54.98,54.98,30,0,RED INTERNET & PHONE CABLE U,1
3,35 - 39,HSN,209.84,209.84,30,0,VF CABLEMAX,1
4,25 - 29,NRW,63.98,63.98,123,0,GIGAZUHAUSE KABEL,1


In [10]:
data.isnull().any()

customer_age_class                   False
region                               False
basket_package_price_disc            False
internet_package_price_disc          False
basket_days_until_contract_end       False
count_previous_cancellations_0_18    False
internet_product_name                False
flag_churn                           False
dtype: bool